In [1]:
import pandas as pd
import numpy as np

# 1. Memuat data training
train_path = '../data/processed/train.csv'
df_train = pd.read_csv(train_path)

# 2. Membuat User-Item Matrix
# Mengubah data tabular menjadi matriks 2 dimensi. 
# Jika user belum berinteraksi dengan item, kita isi dengan angka 0.
user_item_matrix = df_train.pivot(index='user_id', columns='item_id', values='rating').fillna(0)

print(f"Bentuk User-Item Matrix: {user_item_matrix.shape}")
print("Contoh 5 baris dan 5 kolom pertama:")
display(user_item_matrix.iloc[:5, :5])

Bentuk User-Item Matrix: (751, 1616)
Contoh 5 baris dan 5 kolom pertama:


item_id,1,2,3,4,5
user_id,,,,,
1,5.0,3.0,4.0,3.0,0.0
2,4.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0
6,4.0,0.0,0.0,0.0,0.0


In [2]:
from sklearn.decomposition import TruncatedSVD

# 1. Melatih Model SVD (Kita kompres matriks menjadi 20 fitur tersembunyi / latent features)
svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_

# 2. Merekonstruksi Matriks (Mengalikan kembali untuk mendapatkan prediksi skor)
predicted_ratings = np.dot(user_factors, item_factors)

# 3. Membuat tabel baru berisi prediksi skor untuk SEMUA kombinasi user dan item
df_predictions = pd.DataFrame(
    predicted_ratings, 
    index=user_item_matrix.index, 
    columns=user_item_matrix.columns
)

print("Proses training Matrix Factorization selesai!")
print("Contoh prediksi skor untuk 5 baris dan 5 kolom pertama:")
display(df_predictions.iloc[:5, :5])

Proses training Matrix Factorization selesai!
Contoh prediksi skor untuk 5 baris dan 5 kolom pertama:


item_id,1,2,3,4,5
user_id,,,,,
1,4.371636,2.391482,1.633147,2.706351,0.359182
2,2.054456,-0.120548,0.070897,0.317687,0.042465
3,-0.179082,-0.014573,0.116156,-0.105381,0.004121
5,3.344282,1.082684,0.351347,1.891708,0.304186
6,3.117617,0.283377,0.294029,1.460389,-0.713619


In [3]:
def get_cf_recommendations(user_id, top_n=5):
    # Cek apakah user ada di data training (Cold-start check)
    if user_id not in df_predictions.index:
        return {"error": "User tidak ditemukan di data latih (Cold Start)"}
    
    # 1. Ambil semua prediksi skor untuk user ini
    user_scores = df_predictions.loc[user_id]
    
    # 2. Ambil daftar item yang SUDAH pernah berinteraksi dengan user ini
    user_history = df_train[df_train['user_id'] == user_id]['item_id'].tolist()
    
    # 3. Hapus item yang sudah pernah dilihat/dibeli agar tidak direkomendasikan ulang
    user_scores = user_scores.drop(user_history, errors='ignore')
    
    # 4. Urutkan dari skor tertinggi dan ambil Top N
    top_items = user_scores.sort_values(ascending=False).head(top_n)
    
    # Format output
    result = []
    for item_id, score in top_items.items():
        result.append({
            "item_id": int(item_id),
            "predicted_score": round(score, 3)
        })
        
    return {
        "user_id": user_id,
        "recommendation_type": "collaborative_filtering",
        "recommendations": result
    }

# Tes Perbandingan!
print("Rekomendasi Personal untuk User 999:")
print(get_cf_recommendations(user_id=999, top_n=3))

print("\nRekomendasi Personal untuk User 12:")
print(get_cf_recommendations(user_id=12, top_n=3))

Rekomendasi Personal untuk User 999:
{'error': 'User tidak ditemukan di data latih (Cold Start)'}

Rekomendasi Personal untuk User 12:
{'user_id': 12, 'recommendation_type': 'collaborative_filtering', 'recommendations': [{'item_id': 423, 'predicted_score': 2.728}, {'item_id': 64, 'predicted_score': 2.535}, {'item_id': 22, 'predicted_score': 2.252}]}
